<a href="https://colab.research.google.com/github/Tanishk-Rastogi/RNN_Implementation/blob/main/RNN_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [27]:
df = pd.read_csv("https://raw.githubusercontent.com/Tanishk-Rastogi/RNN_Implementation/refs/heads/main/sentiment_analysis.csv")

In [28]:
df.head()

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram


In [14]:
# keras -> builds neural networks
# tokenizer -> converts text into int
# pad seq -> makes every sentence the same len
# model -> create custom model
# embedding, simplernn, dense -> neural network layers

In [15]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]

In [16]:
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [17]:
labels

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [18]:
vocab_size = 2000

# tokenizer converts words into no.
tok = Tokenizer(num_words=vocab_size, oov_token='')

# tokenizer reads every sentence and builds a dict
tok.fit_on_texts(sentences)

# convert the sentences
seqs = tok.texts_to_sequences(sentences)

# different words have different lens so we find the max
maxlen = max(len(s) for s in seqs)

# for extra lens it represents as 0
X = pad_sequences(seqs, maxlen=maxlen, padding='post')
y = labels

In [19]:
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [20]:
# hyperparameters
embed_dim = 16 # 16 dimensional vector

# rnn hidden state has 8 neurons
rnn_units = 8

In [21]:
# input layer
inp = Input(shape= (maxlen,),dtype="int32",name = 'input')

# embedding layer
x = Embedding(input_dim=vocab_size,output_dim=embed_dim,mask_zero=True,name = 'embed')(inp)

# sentence goes word by word
rnn = SimpleRNN(units = rnn_units, return_sequences = False,return_state= False,name = 'simple_rnn')
x_last = rnn(x)

# dense layer
out = Dense(1,activation='sigmoid',name = 'out')(x_last)
model = Model(inputs =inp,outputs = out)

# adam is the activation func
model.compile(optimizer='adam',loss = 'binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(X,y,epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4000 - loss: 0.7035
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 0.6867 
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 0.6730 
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8000 - loss: 0.6610
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9000 - loss: 0.6467
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9333 - loss: 0.6334 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9667 - loss: 0.6188
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9667 - loss: 0.6036
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9667 - loss: 0.5873
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9667 - loss: 0.5699  
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5510 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.531

In [23]:
intermediate_model = Model(inputs=model.inputs, outputs=[model.get_layer('embed').output, model.get_layer('simple_rnn').output])

In [24]:
# we want to inspect what happens inside
# therefore another model was created

from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed')(seq_inp)  # reuse trained embedding

# Create RNN with return_sequences=True
rnn_seq = SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')

# DO NOT CALL build() manually
seq_hidden = rnn_seq(seq_emb)  # builds automatically

# Copy trained RNN weights
try:
    trained_weights = model.get_layer('simple_rnn').get_weights()
    rnn_seq.set_weights(trained_weights)
    print("Copied RNN weights into sequence-inspection RNN.")
except Exception as e:
    print("Could not copy weights automatically:", e)

inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Inspect
idx = 0
example_seq = X[idx:idx+1]  # shape (1, maxlen)
hidden_seq = inspect_model.predict(example_seq)

print("Sentence:", sentences[idx])
print("Token ids:", example_seq)
print("Hidden states per timestep shape:", hidden_seq.shape)
print("Hidden states (timesteps x units):")
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.019  0.035 -0.053 -0.022  0.032 -0.042  0.03   0.001]
 [-0.041  0.021  0.055 -0.046  0.063 -0.258  0.237 -0.017]
 [ 0.117  0.2    0.101  0.014 -0.011 -0.096 -0.175  0.243]
 [ 0.257  0.023 -0.346  0.206 -0.272 -0.16   0.074 -0.051]
 [ 0.257  0.023 -0.346  0.206 -0.272 -0.16   0.074 -0.051]]
